### Simulating a Simple Pendulum with Euler's Method

A pendulum released from rest and left alone has no friction to remove energy
and no motor to add any, so it should swing to the same height forever. That
makes it a demanding test of an integrator: any method that quietly gains or
loses energy will show it as an amplitude that creeps up or dies away, and the
error is impossible to miss.

This notebook applies the simplest method there is, **Forward Euler**, and
watches it fail. The next notebook (`cromer_pendulum.ipynb`) fixes the failure
by changing a single array index.

### From one second-order ODE to two first-order ODEs

The exact equation of motion for a simple pendulum of length $L$ is:

$$\frac{d^2\theta}{dt^2} = -\frac{g}{L}\sin\theta$$

This second-order ODE is split into two coupled first-order ODEs by
introducing the angular velocity $\omega = d\theta/dt$:

$$\frac{d\omega}{dt} = -\frac{g}{L}\sin\theta
\qquad
\frac{d\theta}{dt} = \omega$$

Splitting a higher-order equation into a system of first-order equations is the
standard move that lets any first-order stepper handle it, and every method in
this topic uses the same pair.

### The Forward Euler update

The Forward Euler update at each time step $i$ is:

$$\omega_{i+1} = \omega_i - \frac{g}{L}\sin\theta_i\,\Delta t
\qquad
\theta_{i+1} = \theta_i + \omega_i\,\Delta t$$

Look carefully at the second equation. It advances the angle using
$\omega_{\mathbf{i}}$, the velocity from the *start* of the step, even though
$\omega_{i+1}$ was just computed on the line above. That stale velocity is what
causes the trouble.

Forward Euler does not conserve energy for oscillatory systems, so the amplitude
will drift upward over time. The final cell measures how badly.

The small-angle approximation ($\sin\theta \approx \theta$) gives an analytic
period of $T = 2\pi\sqrt{L/g} \approx 2.006\,\text{s}$ for $L = 1\,\text{m}$. However, at $\theta_0 = 45^\circ$, the true period is slightly longer at $2.086\,\text{s}$,
because the exact restoring torque is weaker than the linear one.

---
### Setup: simulation parameters

The simulation runs for 10 seconds with 500 time steps of
$\Delta t = 0.02\,\text{s}$ (20 ms) each.

Ten seconds covers roughly five full swings, which is long enough to see the
amplitude grow. The same grid is reused in `cromer_pendulum.ipynb` and in
`symplectic_integrators.ipynb` so the plots can be compared side by side.

In [ ]:
"""euler_pendulum.ipynb"""

# Cell 01 - Simulation parameters

%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

tf = 10  # final time (seconds)
ts = 500  # number of time steps
dt = tf / ts  # time step size (0.02 s = 20 ms)

print(f"{tf=:,}  {ts=:,}  {dt=:.3f}")

---
### Pre-allocating the state arrays

Three arrays hold the time stamps $t$, the angular displacement $\theta$
(radians), and the angular velocity $\omega$ (rad/s) at each step. Together
$\theta$ and $\omega$ are the complete **state** of the pendulum: given both at
one instant, the equations of motion determine every instant after it.

Unicode code points are used for the Greek column labels in the DataFrame
preview. The table shows all zeros because the initial conditions are not set
until the next cell.

In [ ]:
# Cell 02 - Allocate arrays for time, angular displacement, and angular velocity
# Unicode: \u03b8 = theta, \u03c9 = omega

t = np.zeros(ts)
omega = np.zeros(ts)
theta = np.zeros(ts)

pd.DataFrame({"t": t[:5], "\u03b8": theta[:5], "\u03c9": omega[:5]})

---
### Initial conditions

The pendulum is released from rest at $\theta_0 = 45^\circ$, so the initial
angular velocity is $\omega_0 = 0$ (already set by `np.zeros`).

This is a deliberately large release angle. At $45^\circ$ the small-angle
approximation $\sin\theta \approx \theta$ is off by about 10%, so the pendulum
is genuinely nonlinear and the simulation cannot be checked against a simple
sine wave. The physical constants ($m = 1\,\text{kg}$, $L = 1\,\text{m}$,
$g = 9.81\,\text{m/s}^2$) are set here as well; mass affects only the energy
bookkeeping in the last cell, never the motion itself.

In [ ]:
# Cell 03 - Initial conditions

mass = 1.0  # pendulum bob mass (kg)
length = 1.0  # pendulum length (m)
g = 9.81  # gravitational acceleration (m/s^2)

theta[0] = np.deg2rad(45)  # initial displacement: 45 degrees -> radians
# omega[0] = 0 already (released from rest)

pd.DataFrame({"t": t[:5], "\u03b8": theta[:5], "\u03c9": omega[:5]})

---
### Forward Euler integration

Both state variables are updated from values taken at the start of the step.

Note that $\theta_{i+1}$ uses $\omega_i$, not $\omega_{i+1}$. This is the
defining characteristic of Forward Euler, and the source of its energy drift:
the angle is advanced with a velocity that the current force has not yet
corrected, so the bob consistently overshoots and gains a little energy every
step.

In [ ]:
# Cell 04 - Forward Euler integration of the pendulum equations of motion

for i in range(ts - 1):
    t[i + 1] = t[i] + dt
    omega[i + 1] = omega[i] - g / length * np.sin(theta[i]) * dt
    theta[i + 1] = theta[i] + omega[i] * dt

pd.DataFrame({"t": t[:5], "\u03b8": theta[:5], "\u03c9": omega[:5]})

---
### Plotting $\theta$ and $\omega$ on a dual y-axis

Angular displacement (left axis) and angular velocity (right axis) are plotted
on the same time axis using `plt.twinx()`.

The growing amplitude in $\theta$ is the energy drift introduced by Forward
Euler. The pendulum starts at $45^\circ$ and by the end of the 10-second run it
is swinging past $98^\circ$, more than twice as far, even though nothing in the
physics is pushing it. This is not rounding error, it is the method itself
manufacturing energy. Halving $\Delta t$ slows the growth but never stops it:
for any fixed step size the amplitude still climbs without bound if you run
long enough.

In [ ]:
# Cell 05 - Dual y-axis plot of angular displacement and angular velocity

fig, ax1 = plt.subplots(figsize=(9, 5))

(plot1,) = ax1.plot(t, theta, lw=2, label=r"$\theta$ (rad)")
ax1.set_xlabel("Time (s)")
ax1.set_ylabel(r"Angular Displacement $\theta$ (rad)")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
(plot2,) = ax2.plot(t, omega, lw=2, color="orange", label=r"$\omega$ (rad/s)")
ax2.set_ylabel(r"Angular Velocity $\omega$ (rad/s)")

plt.title("Simple Pendulum - Euler's Method")
plt.legend(
    [plot1, plot2], [r"$\theta$", r"$\omega$"], framealpha=1.0, facecolor="white"
)
plt.tight_layout()
plt.show()

---
### Measuring the energy drift

The plot shows the problem; this cell puts a number on it. Energy is measured
relative to the lowest point of the swing ($\theta = 0$), so the kinetic and
potential terms are

$$K = \tfrac{1}{2} m L^2 \omega^2
\qquad
U = mgL\left(1 - \cos\theta\right)$$

and the total is $E = K + U$. Released from rest, the pendulum starts with
$K = 0$ and all of its energy in $U$.

For a real pendulum $E$ is constant for all time. Compare the printed initial
and final values: Forward Euler ends the 10-second run with roughly **4.5 times
the energy it started with**, a drift of about **+348%**. That single number is
the entire case against the method, and the next notebook removes almost all of
it by changing one index.

In [ ]:
# Cell 06 - Energy analysis

# Initial Kinetic energy (should be zero at t=0)
Ei = 0.5 * mass * length**2 * omega[0] ** 2
# Initial Potential energy (relative to lowest point of the swing)
Ei += mass * g * length * (1 - np.cos(theta[0]))

# Final Kinetic energy
Ef = 0.5 * mass * length**2 * omega[-1] ** 2
# Final Potential energy
Ef += mass * g * length * (1 - np.cos(theta[-1]))

# Energy drift in joules as a percentage of initial energy
drift_pct = 100 * (Ef - Ei) / Ei

print(f"Initial energy : {Ei:6.3f} J")
print(f"Final energy   : {Ef:6.3f} J")
print(f"Energy drift   : {Ef - Ei:6.3f} J  ({drift_pct:6.3f}%)")

: 